In [ ]:
import os
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.common.config import Config
from src.data.build_normalized_paragraphs import (
    iter_bz2_files,
    make_shard_key_from_path,
    process_single_bz2_file,
)
from src.data.normalized_store_writer import write_manifest

In [ ]:
config = Config()

## 1. process all .bz2 files (parallel option)

In [ ]:
RAW_WIKI_DIR = Path(config.raw_wiki_2017_dir)
OUTPUT_ROOT = PROJECT_ROOT / "data" / "interim" / "normalized_paragraphs" / "notebook_all_shards"
SHARD_OUTPUT_DIR = OUTPUT_ROOT / "shards"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
SUMMARY_PATH = MANIFEST_DIR / "summary.json"

bz2_files = iter_bz2_files(RAW_WIKI_DIR)
min_paragraph_chars = config.min_paragraph_chars
max_valid_docs_per_shard = None  # None -> each .bz2 file is processed to the end

use_parallel = True
max_workers = max(1, (os.cpu_count() or 2) - 1)

summary = []

if use_parallel:
    print(f"Running in parallel with {max_workers} workers")
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {}
        for bz2_path in bz2_files:
            shard_key = make_shard_key_from_path(bz2_path, RAW_WIKI_DIR)
            future = executor.submit(
                process_single_bz2_file,
                bz2_path=bz2_path,
                output_dir=SHARD_OUTPUT_DIR,
                min_paragraph_chars=min_paragraph_chars,
                max_valid_docs=max_valid_docs_per_shard,
                shard_key=shard_key,
            )
            futures[future] = (bz2_path, shard_key)

        for done_idx, future in enumerate(as_completed(futures), start=1):
            bz2_path, shard_key = futures[future]
            stats = future.result()
            summary.append(stats)

            manifest_path = MANIFEST_DIR / f"{shard_key}.json"
            write_manifest(stats, manifest_path)

            print(
                f"[{done_idx}/{len(bz2_files)}] done: {bz2_path.name} | "
                f"valid_docs={stats['valid_docs']} | "
                f"paragraphs={stats['total_paragraph_records_after_dedup']}"
            )
else:
    print("Running in sequential mode")
    for shard_idx, bz2_path in enumerate(bz2_files, start=1):
        shard_key = make_shard_key_from_path(bz2_path, RAW_WIKI_DIR)
        print(f"[{shard_idx}/{len(bz2_files)}] processing: {bz2_path}")

        stats = process_single_bz2_file(
            bz2_path=bz2_path,
            output_dir=SHARD_OUTPUT_DIR,
            min_paragraph_chars=min_paragraph_chars,
            max_valid_docs=max_valid_docs_per_shard,
            shard_key=shard_key,
        )
        summary.append(stats)

        manifest_path = MANIFEST_DIR / f"{shard_key}.json"
        write_manifest(stats, manifest_path)

summary = sorted(summary, key=lambda x: x['source_path'])
summary_payload = {
    "num_shards_processed": len(summary),
    "total_valid_docs": sum(item["valid_docs"] for item in summary),
    "total_paragraph_records_after_dedup": sum(
        item["total_paragraph_records_after_dedup"] for item in summary
    ),
    "shards": summary,
}
write_manifest(summary_payload, SUMMARY_PATH)

summary_df = pd.DataFrame(summary)
print("num_shards_processed:", len(summary_df))
print("total_valid_docs:", summary_payload["total_valid_docs"])
print("total_paragraph_records_after_dedup:", summary_payload["total_paragraph_records_after_dedup"])

sample_output_path = Path(summary_df.iloc[0]["output_path"])
df_check = pd.read_parquet(sample_output_path)
print("sample_output_path:", sample_output_path)
print("sample_shape:", df_check.shape)
summary_df.head(10)


Running in parallel with 7 workers
[1/15517] done: wiki_16.bz2 | valid_docs=47 | paragraphs=1568
[2/15517] done: wiki_26.bz2 | valid_docs=28 | paragraphs=1422
[3/15517] done: wiki_22.bz2 | valid_docs=65 | paragraphs=1820
[4/15517] done: wiki_04.bz2 | valid_docs=25 | paragraphs=1416
[5/15517] done: wiki_01.bz2 | valid_docs=34 | paragraphs=1685
[6/15517] done: wiki_15.bz2 | valid_docs=48 | paragraphs=1674
[7/15517] done: wiki_07.bz2 | valid_docs=49 | paragraphs=1642
[8/15517] done: wiki_09.bz2 | valid_docs=63 | paragraphs=1646
[9/15517] done: wiki_25.bz2 | valid_docs=72 | paragraphs=1862
[10/15517] done: wiki_18.bz2 | valid_docs=56 | paragraphs=1631
[11/15517] done: wiki_10.bz2 | valid_docs=66 | paragraphs=1748
[12/15517] done: wiki_05.bz2 | valid_docs=45 | paragraphs=1659
[13/15517] done: wiki_27.bz2 | valid_docs=37 | paragraphs=1536
[14/15517] done: wiki_11.bz2 | valid_docs=73 | paragraphs=1486
[15/15517] done: wiki_12.bz2 | valid_docs=105 | paragraphs=1841
[16/15517] done: wiki_17.bz2

,source_path,output_path,total_lines_read,valid_docs,skipped_docs,total_paragraph_records_before_dedup,total_paragraph_records_after_dedup,dedup_dropped_count,num_unique_titles,num_unique_doc_ids,num_unique_paragraph_uids
0,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,29,29,0,1476,1476,0,28,28,1476
1,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,34,34,0,1685,1685,0,33,33,1685
2,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,32,32,0,1419,1419,0,30,30,1419
3,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,33,33,0,1484,1484,0,32,32,1484
4,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,25,25,0,1416,1416,0,25,25,1416
5,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,45,45,0,1659,1659,0,44,44,1659
6,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,51,51,0,1775,1775,0,51,51,1775
7,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,49,49,0,1642,1642,0,49,49,1642
8,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,41,41,0,1736,1736,0,39,39,1736
9,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,d:\AI\projects\multi-hop-rag-hover\data\interi...,63,63,0,1646,1646,0,62,62,1646


## 2. data sanity check

In [4]:
print("summary null counts")
print(summary_df.isnull().sum())
print()

print("summary total rows              :", len(summary_df))
print("summary total valid docs        :", summary_df["valid_docs"].sum())
print("summary total skipped docs      :", summary_df["skipped_docs"].sum())
print("summary total output paragraphs :", summary_df["total_paragraph_records_after_dedup"].sum())
print()

print("sample shard null counts")
print(df_check.isnull().sum())
print()

print("sample unique doc_id count        :", df_check["doc_id"].nunique())
print("sample unique title_norm count    :", df_check["title_norm"].nunique())
print("sample unique paragraph_id count  :", df_check["paragraph_id"].nunique())
print("sample unique paragraph_uid count :", df_check["paragraph_uid"].nunique())
print()

print("sample duplicate paragraph_uid rows:", df_check.duplicated(subset=["paragraph_uid"]).sum())
print("sample duplicate paragraph_id rows :", df_check.duplicated(subset=["paragraph_id"]).sum())


summary null counts
source_path                             0
output_path                             0
total_lines_read                        0
valid_docs                              0
skipped_docs                            0
total_paragraph_records_before_dedup    0
total_paragraph_records_after_dedup     0
dedup_dropped_count                     0
num_unique_titles                       0
num_unique_doc_ids                      0
num_unique_paragraph_uids               0
dtype: int64

summary total rows              : 15517
summary total valid docs        : 5486212
summary total skipped docs      : 0
summary total output paragraphs : 32165896

sample shard null counts
doc_id                0
title                 0
title_norm            0
paragraph_id          0
paragraph_uid         0
paragraph_index       0
paragraph_text_raw    0
paragraph_text        0
source_path           0
record_id             0
dtype: int64

sample unique doc_id count        : 28
sample unique title_norm